In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from joblib import load,dump
import joblib
import warnings

In [2]:
X_train, X_test, y_train, y_test = load("../train_test_split.pkl")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (72840, 27)
X_test shape: (18211, 27)
y_train shape: (72840,)
y_test shape: (18211,)


# Models

## XGB + Optuna



In [3]:
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def xgb_objective(trial):
    params = {
        # Number of boosting rounds
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        
        # Maximum depth of each tree
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        
        # Learning rate
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        
        # Subsampling ratio of the training data
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        
        # Subsampling ratio of columns for each tree
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        
        # Minimum sum of instance weight needed in a child
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        
        # L2 regularization term on weights
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        
        # L1 regularization term on weights
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        
        # Handle class imbalance
        'scale_pos_weight': y_train.value_counts()[0] / y_train.value_counts()[1],
        
        'random_state': 42,
        'use_label_encoder': False,
        'eval_metric': 'logloss',
    }

    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='roc_auc', n_jobs=-1)
    return np.mean(scores) 

# Create Optuna study
sampler = optuna.samplers.TPESampler(seed=42)

study = optuna.create_study(
    direction='maximize',
    sampler=sampler
)
study.optimize(xgb_objective, n_trials=150)  


print("Best XGB params:", study.best_params)
print("Best CV AUC:", study.best_value)


xgb_best = XGBClassifier(**study.best_params,
                         scale_pos_weight=y_train.value_counts()[0]/y_train.value_counts()[1],
                         random_state=42,
                         use_label_encoder=False,
                         eval_metric='logloss')
xgb_best.fit(X_train, y_train)

# Evaluate on test set
from sklearn.metrics import roc_auc_score
y_proba = xgb_best.predict_proba(X_test)[:,1]
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))

[I 2026-03-29 23:08:10,298] A new study created in memory with name: no-name-d3c81ded-c8e8-4a11-9f28-480a4bb3697d
[I 2026-03-29 23:08:22,284] Trial 0 finished with value: 0.9486044111718309 and parameters: {'n_estimators': 437, 'max_depth': 15, 'learning_rate': 0.1205712628744377, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 2, 'reg_lambda': 0.2904180608409973, 'reg_alpha': 4.330880728874676}. Best is trial 0 with value: 0.9486044111718309.
[I 2026-03-29 23:08:38,228] Trial 1 finished with value: 0.9486499845796116 and parameters: {'n_estimators': 641, 'max_depth': 12, 'learning_rate': 0.010725209743171996, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'min_child_weight': 3, 'reg_lambda': 0.9091248360355031, 'reg_alpha': 0.9170225492671691}. Best is trial 1 with value: 0.9486499845796116.
[I 2026-03-29 23:08:45,677] Trial 2 finished with value: 0.9522799027873384 and parameters: {'n_estimators': 374, 'max_depth'

Best XGB params: {'n_estimators': 572, 'max_depth': 5, 'learning_rate': 0.09169666705394995, 'subsample': 0.9885722540865319, 'colsample_bytree': 0.5628115781614443, 'min_child_weight': 10, 'reg_lambda': 4.43966161008723, 'reg_alpha': 0.7545713777040758}
Best CV AUC: 0.9553240617855856
Test ROC AUC: 0.9590115342406336


In [4]:

joblib.dump(study.best_params, "../best_xgb_params.pkl")

['../best_xgb_params.pkl']